# Chapter 34 — SDK Sessions

**Companion to *Pi Agents*** · [`Pi Agents` chapter 34](https://programmer.ie/books/pi/34-chapter/)

| | |
|---|---|
| Chapter | `34-chapter.md` · weight 34 · `/books/pi/34-chapter/` |
| Notebook | `notebooks/pi/34-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does assigning `agent.state.messages` change what the provider receives?

The chapter's point: the `SessionManager` is **authoritative**. The in-memory
`agent.state.messages` is a view; assigning to it does not change what the
provider is sent. The session manager's history does.


## What this notebook establishes

- Streaming: subscribe **before** prompting, and text arrives as typed `text_delta` events; `getLastAssistantText()` returns the same text.
- A prompt sent while streaming **must say steer or follow-up**, or it rejects.
- `SessionManager` is **authoritative**: assigning `agent.state.messages` does not change what is sent.
- An in-memory `SessionManager` **writes no file** (`getSessionFile()` is undefined).
- `dispose()` **stops event delivery** — a disposed session delivers nothing to listeners.

## What this notebook does **not** establish

- **No latency, throughput or isolation-failure measurement** (`ch34-lim1`).
- **The chapter's `createAgentSession` snippets are `documented-not-run`** — the harness uses `createAgentSession` internally but the chapter's snippets are not executed here.
- **The model is scripted** — this measures the session around the model, not model behaviour.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real `AgentSession`s through the book's harness.
* **Evidence scope:** `in_process_runtime`. The session authority, streaming and disposal are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(34))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/34-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: streaming and the session authority

In [3]:
SDK = pinb.driver_source("ch34-sdk.ts")
sdk = pinb.driver(SDK, label="ch34-sdk", timeout=300)
s = sdk["streaming"]
print(pinb.md_table(
    ["check", "value"],
    [["deltas received", s["deltas"]],
     ["buffered text", s["joined"]],
     ["getLastAssistantText", s["last"]]],
))

| check | value |
|---|---|
| deltas received | 3 |
| buffered text | the repository has three packages |
| getLastAssistantText | the repository has three packages |


In [4]:
pinb.show_checks([
    pinb.check("text arrives across more than one delta",
               s["deltas"] > 1, f"{s['deltas']} deltas"),
    pinb.check("the deltas concatenate to the full answer",
               s["joined"] == "the repository has three packages", "joined ok"),
    pinb.check("getLastAssistantText matches the buffered text",
               s["last"] == s["joined"], "same text"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------  ---------
PASS  text arrives across more than one delta  3 deltas
PASS  the deltas concatenate to the full answer  joined ok
PASS  getLastAssistantText matches the buffered text  same text

3/3 assertions held.


## Prompting while streaming requires steer or follow-up

In [5]:
b = sdk["streamingBehaviour"]
print(pinb.md_table(
    ["check", "value"],
    [["was streaming", b["wasStreaming"]],
     ["prompt without behaviour rejected", b["rejected"]],
     ["steer disposition", b["steered"]],
     ["followUp disposition", b["followed"]]],
))

| check | value |
|---|---|
| was streaming | True |
| prompt without behaviour rejected | True |
| steer disposition | queued |
| followUp disposition | queued |


In [6]:
pinb.show_checks([
    pinb.check("the session was streaming", b["wasStreaming"] is True, "streaming"),
    pinb.check("a prompt with no behaviour is rejected", b["rejected"], "rejected"),
    pinb.check("steer is queued", b["steered"] == "queued", "queued"),
    pinb.check("followUp is queued", b["followed"] == "queued", "queued"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------  ---------
PASS  the session was streaming  streaming
PASS  a prompt with no behaviour is rejected  rejected
PASS  steer is queued  queued
PASS  followUp is queued  queued

4/4 assertions held.


## SessionManager is authoritative

Overwrite the in-memory transcript, then send another prompt. The forged history
never reaches the provider; the session manager's real history does.

In [7]:
a = sdk["authoritative"]
print(pinb.md_table(
    ["check", "value"],
    [["forged history reached the provider", a["sentHasForged"]],
     ["real history reached the provider", a["sentHasReal"]]],
))

| check | value |
|---|---|
| forged history reached the provider | False |
| real history reached the provider | True |


In [8]:
pinb.show_checks([
    pinb.check("the forged transcript never reached the provider",
               not a["sentHasForged"], "forged absent"),
    pinb.check("the real history, from the session manager, did",
               a["sentHasReal"], "real present"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------------  -------------
PASS  the forged transcript never reached the provider  forged absent
PASS  the real history, from the session manager, did  real present

2/2 assertions held.


## In-memory writes no file; dispose stops delivery

In [9]:
d = sdk["dispose"]
print(pinb.md_table(
    ["check", "value"],
    [["in-memory session file is undefined", sdk["inMemory"]["fileUndefined"]],
     ["events before dispose", d["eventsBefore"]],
     ["events after dispose", d["eventsAfter"]]],
))

| check | value |
|---|---|
| in-memory session file is undefined | True |
| events before dispose | 0 |
| events after dispose | 0 |


In [10]:
pinb.show_checks([
    pinb.check("an in-memory SessionManager writes no file",
               sdk["inMemory"]["fileUndefined"], "no file"),
    pinb.check("a disposed session delivers nothing to listeners",
               d["stopped"], f"{d['eventsBefore']} -> {d['eventsAfter']}"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------------  --------
PASS  an in-memory SessionManager writes no file  no file
PASS  a disposed session delivers nothing to listeners  0 -> 0

2/2 assertions held.


## The chapter's own tests

All five SDK tests.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [11]:
PATTERNS = 'ch34-sdk/sdk.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch34-sdk/sdk.test.ts', show="session")

$ node --test ch34-sdk/sdk.test.ts
  PASS  5 passed, 0 failed, 0 skipped  in 1983 ms

  tests matching 'session':
    ok   a prompt sent while the session is streaming must say steer or follow-up, or it rejects
    ok   SessionManager is authoritative: assigning agent.state.messages does not change what is sent
    ok   an in-memory SessionManager writes no file


## Your turn: predict, then run

**Predict first.** The forged `agent.state.messages` did not reach the provider.
What *does* the `agent.state.messages` view return after the second prompt — the
forged array, or the session manager's history?

**Then change one input.** Use a persistent `SessionManager` (`persist: true`).
Does `getSessionFile()` return a path? Does `dispose()` delete the file?

**Predict the boundary.** `dispose()` stops delivery to listeners already
subscribed. What happens to a *new* subscriber added after dispose?

In [12]:
print("Predictions:")
print("  1. agent.state.messages returns the session manager's history after the next prompt.")
print("  2. Persistent: getSessionFile() returns a path; dispose() does not delete the file.")
print("  3. A new subscriber after dispose receives nothing.")
print()
print("Observed above:")
print(f"  streaming: deltas={s['deltas']}, joined matches last={s['joined'] == s['last']}")
print(f"  streaming behaviour: rejected={b['rejected']}, steer={b['steered']}, followUp={b['followed']}")
print(f"  authority: forged={a['sentHasForged']}, real={a['sentHasReal']}")
print(f"  in-memory file undefined={sdk['inMemory']['fileUndefined']}, dispose stopped={d['stopped']}")
print()
assert s["deltas"] > 1 and s["joined"] == s["last"]
assert b["rejected"] and b["steered"] == "queued" and b["followed"] == "queued"
assert not a["sentHasForged"] and a["sentHasReal"]
assert sdk["inMemory"]["fileUndefined"] and d["stopped"]
print("held: SessionManager is authoritative; streaming needs a behaviour; dispose stops delivery")

Predictions:
  1. agent.state.messages returns the session manager's history after the next prompt.
  2. Persistent: getSessionFile() returns a path; dispose() does not delete the file.
  3. A new subscriber after dispose receives nothing.

Observed above:
  streaming: deltas=3, joined matches last=True
  streaming behaviour: rejected=True, steer=queued, followUp=queued
  authority: forged=False, real=True
  in-memory file undefined=True, dispose stopped=True

held: SessionManager is authoritative; streaming needs a behaviour; dispose stops delivery


## Interpretation

The SDK session has three contracts, and the chapter shows each:

| Contract | What it means | The boundary |
|---|---|---|
| **Session authority** | `SessionManager` decides what is sent | `agent.state.messages` is a view, not the source |
| **Streaming** | Subscribe first; deltas arrive typed | A prompt mid-stream needs `steer` or `follow-up` |
| **Disposal** | `dispose()` stops delivery | Already-subscribed listeners receive nothing after |

Practical rules:
1. **The session manager is the source of truth** — do not mutate the transcript and expect it to stick.
2. **Subscribe before prompting** — a subscriber added after the deltas begin misses them.
3. **Say `steer` or `follow-up`** — a bare prompt mid-stream is an error by design.
4. **`dispose()` when done** — it stops event delivery; an in-memory session leaves no file.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real `AgentSession`s through the book's harness.

**Evidence scope:** `in_process_runtime`. The session authority, streaming and disposal are Pi's.

### What was read or run

- **Ran** `drivers/ch34-sdk.ts`: real sessions covering streaming, streaming behaviour, session authority, in-memory sessions, and disposal.
- **Ran** `ch34-sdk/sdk.test.ts` (5 tests).
- **Read** `examples/evidence.json`, chapter 34 rows (5 claims).

### Limitations

- **No latency, throughput or isolation-failure measurement** (`ch34-lim1`).
- **The chapter's `createAgentSession` snippets are `documented-not-run`**.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
